# Notebook 06: Explainable Risk Scoring, Rules & Anomaly Detection

This notebook demonstrates the transparent risk quantification engine:
- 0–100 weighted risk score
- Low / Medium / High classification
- Severe clause term identification with page references
- Date chronology contradiction detection
- Jurisdictional conflict checks
- Missing essential clause penalties.

In [ ]:
import os
import sys
import json
import pandas as pd

sys.path.insert(0, os.path.abspath(os.path.join("..", "..", "Backend")))

from ml.preprocessing.document_pipeline import process_contract_document
from ml.classifiers.clause_classifier import LegalClauseClassifier
from ml.ner.entity_extractor import LegalEntityExtractor
from ml.risk.risk_engine import LegalRiskEngine

DATA_DIR = os.path.abspath(os.path.join("..", "data", "sample_contracts"))
high_risk_pdf = os.path.join(DATA_DIR, "sample_high_risk_contract.pdf")

with open(high_risk_pdf, "rb") as f:
    doc_result = process_contract_document(f.read(), filename="sample_high_risk_contract.pdf")

classifier = LegalClauseClassifier()
extractor = LegalEntityExtractor()
risk_engine = LegalRiskEngine()

classified_chunks = classifier.classify_chunks(doc_result["chunks"])
entities = extractor.extract_all_entities(doc_result["full_text"])
risk_report = risk_engine.evaluate_contract_risk(doc_result, classified_chunks, entities)

print(f"Risk Score: {risk_report['score']} / 100 | Risk Level: {risk_report['level']}")
print(f"Total Inconsistencies Detected: {len(risk_report['inconsistencies'])}")

## 1. Inspect Risk Factors & Evidence Breakdown

In [ ]:
df_reasons = pd.DataFrame(risk_report["reasons"])
df_reasons[["title", "severity", "weight", "page", "evidence"]]